# Fallacy LoRA training on Colab (v4: fixed 10 epochs, no early stopping)
Runtime → Change runtime type → **T4 GPU**. 

`--val-per-label 0` turns off the validation holdout and early stopping, so both runs train on all 2,199 examples for exactly 10 epochs and keep the final adapter. These are directly comparable to A and B (same data, different epochs).
Data now comes from the private Hugging Face dataset `sreejithb/fallacy-dataset` (version `v2`), not from a zip.

**One-time Colab setup:** left sidebar → key icon (Secrets) → Add new secret, name `HF_TOKEN`, value = a Hugging Face **read** token, and switch on *Notebook access*. Upload `fallacy_code.zip` (code only, tiny) to the Files sidebar.


In [ ]:
!nvidia-smi -L
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm

In [ ]:
# Needs fallacy_code.zip in /content (upload it first) and a Colab secret named HF_TOKEN with notebook access on.
!rm -rf /content/work && mkdir -p /content/work && unzip -oq /content/fallacy_code.zip -d /content/work
%cd /content/work
!pip -q install huggingface_hub
!python fetch_data.py --rev v2
!ls . data data/generated

## Run E: rank 16, 10 epochs

In [ ]:
!python train.py --out out/lora_E --epochs 10 --val-per-label 0 --rank 16 2>&1 | tee train_E.log

In [ ]:
!python evaluate.py --adapter out/lora_E --out lora_E_preds.jsonl

## Run F: rank 32, 10 epochs

In [ ]:
!python train.py --out out/lora_F --epochs 10 --val-per-label 0 --rank 32 2>&1 | tee train_F.log

In [ ]:
!python evaluate.py --adapter out/lora_F --out lora_F_preds.jsonl

In [ ]:
!zip -qr lora_result_v4.zip out data/lora_E_preds.jsonl data/lora_F_preds.jsonl train_E.log train_F.log
from google.colab import files
files.download('/content/work/lora_result_v4.zip')